In [ ]:
import glob
import os
import random
import time
import numpy as np
import time
from string import Template
import json
import tempfile

import ray
import supersuit as ss
from stable_baselines3 import PPO
from stable_baselines3 import DQN
from stable_baselines3.common.callbacks import BaseCallback
from stable_baselines3.ppo import CnnPolicy, MlpPolicy
# from stable_baselines3.common.evaluation import evaluate_policy
# from stable_baselines3.dqn import CnnPolicy, MlpPolicy
from sb3_contrib import RecurrentPPO

#from pettingzoo.mpe import simple_tag_v3
import tag

from ray import train, tune
from ray.train import Checkpoint
from ray.tune.schedulers import PopulationBasedTraining, PopulationBasedTrainingReplay

In [ ]:
class MetricLogger(BaseCallback):
    def __init__(self, log_frequency=100, verbose=0, checkpoint_path=None, cp_model=None, model_file_name=None):
        super(MetricLogger, self).__init__(verbose=verbose)
        self.verbose=verbose
        self.log_frequency = log_frequency
        self.train_losses=[]
        self.train_value_losses=[]
        self.checkpoint_path = checkpoint_path
        self.cp_model = cp_model
        self.model_file_name = model_file_name

    def _on_step(self) -> bool:
        if self.n_calls % self.log_frequency == 0:
            if (self.verbose == 1):
                # print(f"iterations: {self.model.logger.name_to_value['train/n_updates']}")
                # print(f"ep_rew_mean: {self.model.logger.name_to_value['train/ep_rew_mean']}")
                # print(f"policy_loss: {self.model.logger.name_to_value['train/policy_loss']}")
                # print(f"value_loss: {self.model.logger.name_to_value['train/value_loss']}")
                # print(f"entropy_loss: {self.model.logger.name_to_value['train/entropy_loss']}")
                # print("--------------------------------")
                with tempfile.TemporaryDirectory() as checkpoint:
                    self.cp_model.save(os.path.join(checkpoint, self.model_file_name))
                    train.report({
                        "loss" : self.model.logger.name_to_value['train/loss'],
                        "policy_gradient_loss" : self.model.logger.name_to_value['train/policy_gradient_loss'],
                        "value_loss" : self.model.logger.name_to_value['train/value_loss'],
                    }, checkpoint=Checkpoint.from_directory(checkpoint))

        return True

In [ ]:
models = {
    "PPO" : PPO,
    "RecPPO" : RecurrentPPO,
    "DQN" : DQN,
}
filename = Template("${env}_mo"
                    "del-${model}_policy-${policy}_tsteps-${timesteps}_${timestamp}")

In [ ]:
def train_model(config):
    tb_logs_path = "../tb_logs"

    env_kwargs = dict(num_good=1, num_adversaries=3, num_obstacles=2, max_cycles=100, continuous_actions=False)
    # env = simple_tag_v3.parallel_env(**env_kwargs)
    env = tag.parallel_env(**env_kwargs)
    env.reset(seed=0)

    print(f"Starting training on {str(env.metadata['name'])}, {config['model_name']}, {config['model_policy']}, {config['steps']}.")

    env = ss.multiagent_wrappers.pad_observations_v0(env)
    env = ss.pettingzoo_env_to_vec_env_v1(env)
    env = ss.concat_vec_envs_v1(env, 8, num_cpus=4, base_class="stable_baselines3")

    model = models[config['model_name']](
        config['model_policy'],
        env,
        verbose=1,
        tensorboard_log=tb_logs_path,
        **config['model_kwargs'],
        device="cuda:0"
    )

    # consider a checkpoint here
    checkpoint = train.get_checkpoint()
    if checkpoint:
        with checkpoint.as_directory() as checkpoint_dir:
            model = model.load(max(glob.glob(f"{checkpoint_dir}/*.zip"), key=os.path.getctime), device="cuda:0")
            model.set_env(env)
            print("########### Loaded checkpoint! ############")

    path = filename.substitute(
        env=env.unwrapped.metadata.get('name'),
        model=config['model_name'],
        policy=config['model_policy'],
        timesteps=config['steps'],
        timestamp=time.strftime('%Y%m%d-%H%M%S'),
    )
    metric_logger = MetricLogger(verbose=1, cp_model=model, model_file_name=path, log_frequency=5000)

    model.learn(total_timesteps=config['steps'], tb_log_name=path, callback=metric_logger)
    # model.save(os.path.join(tmpdir, path))

    print("Model has been saved.")
    print(f"Finished training on {str(env.unwrapped.metadata['name'])}.")

In [ ]:
if ray.is_initialized():
    ray.shutdown()
ray.init(
    ignore_reinit_error=True,
)

ppo_scheduler = PopulationBasedTraining(
    time_attr="training_iteration",
    perturbation_interval=5,
    metric="loss",
    mode="min",
    hyperparam_mutations={
        "steps": [200_000, 500_000, 1_000_000, 2_000_000, 5_000_000],
        "model_kwargs": dict(
            n_steps=tune.randint(32, 5000), # Horizon range (32,5000), PPO recommended n_steps*n_envs
            batch_size=tune.randint(32, 4096), #
            n_epochs=tune.randint(3, 30),
            clip_range=tune.choice([0.1, 0.2, 0.3]),
            target_kl=tune.uniform(0.003, 0.03), # loguniform?
            gae_lambda=tune.uniform(0.9, 1), # .95 default
            vf_coef=tune.loguniform(0.5, 1),
            ent_coef=tune.loguniform(1e-10, 0.01),
            learning_rate=tune.loguniform(5e-6, 0.003)
        )
    }
)

ppo_tuner = tune.Tuner(
    tune.with_resources(
        train_model,
        resources={"gpu": 1}
    ),
    run_config=train.RunConfig(
        name="TagPPO_2",
        stop={"training_iteration": 10},
        checkpoint_config=train.CheckpointConfig(
            checkpoint_score_attribute="loss",
        ),
    ),
    tune_config=tune.TuneConfig(
        scheduler=ppo_scheduler,
        num_samples=4
    ),
    param_space={
        "steps" : 2000000,
        "model_name" : "PPO",
        "model_policy" : "MlpPolicy",
        "model_kwargs" : dict(
            n_steps=np.random.randint(32, 5000), # Horizon range (32,5000), PPO recommended n_steps*n_envs
            batch_size=np.random.randint(32, 4096), #
            n_epochs=np.random.randint(3, 30),
            clip_range=tune.choice([0.1, 0.2, 0.3]),
            target_kl=tune.uniform(0.003, 0.03), # loguniform?
            gae_lambda=tune.uniform(0.9, 1), # .95 default
            vf_coef=tune.loguniform(0.5, 1),
            ent_coef=tune.loguniform(1e-10, 0.01),
            learning_rate=tune.loguniform(5e-6, 0.003)
        )
    }
)

rec_ppo_scheduler = PopulationBasedTraining(
    time_attr="training_iteration",
    perturbation_interval=5,
    metric="loss",
    mode="min",
    hyperparam_mutations={
        "steps": [200_000, 500_000, 1_000_000, 2_000_000, 5_000_000],
        "model_kwargs": dict(
            n_steps=tune.randint(32, 5000), # Horizon range (32,5000), PPO recommended n_steps*n_envs
            batch_size=tune.randint(32, 4096), #
            n_epochs=tune.randint(3, 30),
            clip_range=tune.choice([0.1, 0.2, 0.3]),
            target_kl=tune.uniform(0.003, 0.03), # loguniform?
            gae_lambda=tune.uniform(0.9, 1), # .95 default
            vf_coef=tune.loguniform(0.5, 1),
            ent_coef=tune.loguniform(1e-10, 0.01),
            learning_rate=tune.loguniform(5e-6, 0.003)
        )
    }
)

rec_ppo_tuner = tune.Tuner(
    tune.with_resources(
        train_model,
        resources={"gpu": 1}
    ),
    run_config=train.RunConfig(
        name="TagRecPPO",
        stop={"training_iteration": 10},
        checkpoint_config=train.CheckpointConfig(
            checkpoint_score_attribute="loss",
        ),
    ),
    tune_config=tune.TuneConfig(
        scheduler=rec_ppo_scheduler,
        num_samples=4
    ),
    param_space={
        "steps" : 2000000,
        "model_name" : "RecPPO",
        "model_policy" : "MlpLstmPolicy",
        "model_kwargs" : dict(
            n_steps=np.random.randint(32, 5000), # Horizon range (32,5000), PPO recommended n_steps*n_envs
            batch_size=np.random.randint(32, 4096), #
            n_epochs=np.random.randint(3, 30),
            clip_range=tune.choice([0.1, 0.2, 0.3]),
            target_kl=tune.uniform(0.003, 0.03), # loguniform?
            gae_lambda=tune.uniform(0.9, 1), # .95 default
            vf_coef=tune.loguniform(0.5, 1),
            ent_coef=tune.loguniform(1e-10, 0.01),
            learning_rate=tune.loguniform(5e-6, 0.003)
        )
    }
)

dqn_scheduler = PopulationBasedTraining(
    time_attr="training_iteration",
    perturbation_interval=5,
    metric="loss",
    mode="min",
    hyperparam_mutations={
        "steps": [200_000, 500_000, 1_000_000, 2_000_000, 5_000_000],
        "model_kwargs": dict(
            batch_size=tune.randint(32, 4096),
            buffer_size=tune.choice([500_000, 1_000_000, 5_000_000, 10_000_000]),
            learning_starts=tune.choice([25_000, 50_000, 100_000]),
            target_update_interval=tune.choice([10, 10_000, 50_000, 100_000]),
            gamma=tune.uniform(0.75, 0.99),
            tau=tune.uniform(0.95, 1),
            train_freq=tune.randint(1,10),
            exploration_final_eps=tune.uniform(0.01, 0.08),
            learning_rate=tune.loguniform(5e-6, 0.001)
        )
    }
)

dqn_tuner = tune.Tuner(
    tune.with_resources(
        train_model,
        resources={"gpu": 1}
    ),
    run_config=train.RunConfig(
        name="TagDQN",
        stop={"training_iteration": 10},
        checkpoint_config=train.CheckpointConfig(
            checkpoint_score_attribute="loss",
        ),
    ),
    tune_config=tune.TuneConfig(
        scheduler=rec_ppo_scheduler,
        num_samples=4
    ),
    param_space={
        "steps" : 2000000,
        "model_name" : "DQN",
        "model_policy" : "MlpPolicy",
        "model_kwargs" : dict(
            batch_size=np.random.randint(32, 4096),
            buffer_size=np.random.choice([500_000, 1_000_000, 5_000_000, 10_000_000]),
            learning_starts=np.random.choice([25_000, 50_000, 100_000]),
            target_update_interval=np.random.choice([10, 10_000, 50_000, 100_000]),
            gamma=np.random.uniform(0.75, 0.99),
            tau=np.random.uniform(0.95, 1),
            train_freq=np.random.randint(1,10),
            exploration_final_eps=np.random.uniform(0.01, 0.08),
            learning_rate=tune.loguniform(5e-6, 0.001)
        )
    }
)

In [ ]:
# results_grid = ppo_tuner.fit()
# results_grid = rec_ppo_tuner.fit()
results_grid = dqn_tuner.fit()

In [ ]:
print(f"Best config:\n {results_grid}")

In [9]:
def _eval(config, policy_path):
    render_mode = 'human'
    env_kwargs = dict(num_good=1, num_adversaries=3, num_obstacles=2, max_cycles=100, continuous_actions=False, render_mode=render_mode)
    # env = simple_tag_v3.env(**env_kwargs)
    env = tag.env(**env_kwargs)
    env.metadata['render_fps'] = 60

    num_games = 6

    model = models[config['model_name']].load(policy_path, device="cuda:0")
    rewards = {agent: 0 for agent in env.possible_agents}

    for i in range(num_games):
        env.reset(seed=i)
        env.action_space(env.possible_agents[0]).seed(i)

        for agent in env.agent_iter():
            obs, reward, termination, truncation, info = env.last()
            if agent == 'agent_0':
                obs=np.append(obs, [0,0])
            #print(obs)
            if render_mode== 'human':
                time.sleep(0.01)
            for agent in env.agents:
                rewards[agent] += env.rewards[agent]

            if termination or truncation:
                break
            else:
                if agent == env.possible_agents[0]:
                    act = env.action_space(agent).sample()
                else:
                    act = model.predict(obs, deterministic=True)[0]
            env.step(act)
    env.close()

    avg_reward = sum(rewards.values()) / len(rewards.values())
    avg_reward_per_agent = {
        agent: rewards[agent] / num_games for agent in env.possible_agents
    }
    print(f"Avg reward: {avg_reward}")
    print("Avg reward per agent, per game: ", avg_reward_per_agent)
    print("Full rewards: ", rewards)


In [ ]:
# glob.glob(f"{results_grid[0].path}")

In [ ]:
best_result = results_grid[0]
for result in results_grid:
    if result.metrics['policy_gradient_loss'] < best_result.metrics['policy_gradient_loss']:
        best_result = result

eval_config = None
policy_path = max(
    glob.glob(f"{best_result.checkpoint.path}/*.zip"), key=os.path.getctime
)
with open(f"{best_result.path}/params.json") as params:
    eval_config = json.load(params)
print(best_result.checkpoint.path)

In [ ]:
_eval(eval_config, policy_path=policy_path)

In [15]:
policy_path = "P:/IK_temp/CollInt/best trains/RecPPO"
zip_path = f"{policy_path}/simple_tag_v3_model-RecPPO_policy-MlpLstmPolicy_tsteps-2000000_20231206-164038.zip"

eval_config = None
with open(f"{policy_path}/params.json") as path:
    eval_config = json.load(path)

_eval(eval_config, policy_path=zip_path)

Avg reward: -419.5629185646426
Avg reward per agent, per game:  {'adversary_0': -23.77431116563487, 'adversary_1': -23.77431116563487, 'adversary_2': -23.77431116563487, 'agent_0': -208.38567887952377}
Full rewards:  {'adversary_0': -142.64586699380922, 'adversary_1': -142.64586699380922, 'adversary_2': -142.64586699380922, 'agent_0': -1250.3140732771426}


In [ ]:
# train_model(
#     eval_config
# )

In [ ]:
# trained_path = max(
#     glob.glob(f"*.zip"), key=os.path.getctime
# )
# _eval(eval_config, policy_path=trained_path)